# CME538 - Introduction to Data Science

## Tutorial 2 - Pandas

This tutorial reinforces the Pandas concepts covered in **Lectures 2.1–2.3**.

### Goals

By the end of this tutorial, you should be able to:

- Create and inspect Pandas DataFrames
- Work with Series and Index objects
- Select data using `[]`, `.loc[]`, `.iloc[]`, and Boolean filtering
- Import and explore CSV data
- Use common string methods with `.str`
- Group and aggregate data with `.groupby()`
- Use lambda functions and `.apply()`
- Understand why vectorization is generally preferred over row-by-row iteration
- Combine DataFrames using `pd.concat()` and `.merge()`

### Tutorial Structure

1. DataFrames and indexing
2. Importing and exploring data
3. Filtering and string methods
4. Grouping and aggregation
5. Lambda functions and `.apply()`
6. Iteration and vectorization
7. Concatenating and merging DataFrames
8. Practice challenge

## Setup Notebook

At the start of a notebook, we import the Python packages we plan to use.

- **NumPy** provides numerical arrays and mathematical operations.
- **Pandas** is the main library for tabular data manipulation in this tutorial.
- **Seaborn** provides a high-level interface for statistical visualization.
- **Matplotlib** provides the underlying plotting interface and fine-grained control over figures.

We may use Seaborn and Matplotlib for quick exploratory plots. In modern Jupyter environments, `%matplotlib inline` is normally unnecessary because inline plotting is already the default.

We will also avoid `warnings.filterwarnings("ignore")` unless there is a specific warning we want to address: hiding all warnings can conceal useful information when students are learning.

In [1]:
# Import NumPy for numerical operations
import numpy as np

# Import Pandas for working with Series and DataFrames
import pandas as pd

# Import Seaborn for statistical data visualization
import seaborn as sns

# Import Matplotlib's pyplot module for creating and customizing plots
import matplotlib.pyplot as plt

## 1. Basics

### 1.1. Anatomy of a DataFrame

Pandas provides two fundamental data structures:

- **Series** — a one-dimensional labeled array.
- **DataFrame** — a two-dimensional labeled table made up of rows and columns.

A DataFrame can be thought of as a collection of Series that share the same index.

The following diagram illustrates the relationship between a Series and a DataFrame.

![DFvsSeries](https://storage.googleapis.com/lds-media/images/series-and-dataframe.width-1200.png)
<center>Series and DataFrames: Number of purchases for apples and oranges</center>

https://www.learndatasci.com/tutorials/python-pandas-tutorial-complete-introduction-for-beginners/

### 1.2 Creating DataFrames from Scratch and Selecting Values

There are several ways to create a `DataFrame`. One common approach is to first create a Python dictionary and then pass it to `pd.DataFrame()`.

A Python dictionary stores data as **key-value pairs**:

key → value

In this example:
- Each key will become a DataFrame column.
- Each value will provide the data for that column.
- The values are stored in lists, with one value for each row.
- Dictionaries are written using curly brackets {}.

### Creating a dictionary

Let's create a dictionary containing the number of apples and oranges purchased by four customers.

In [30]:
# Create a dictionary containing the number of apples and oranges purchased by each customer
data = {
    'apples': [3, 2, 0, 1],
    'oranges': [0, 3, 7, 2]
        }

#### Accessing Values in a Dictionary

We can access a value in a dictionary by using its key inside square brackets `[]`.

For example, `data['apples']` retrieves the list associated with the `'apples'` key.

Let's print the result.

In [31]:
# Access the values stored under the 'apples' key
print(data['apples'])

[3, 2, 0, 1]


#### Creating a DataFrame from a Dictionary

We can use the dictionary to create a Pandas `DataFrame`.

When we pass a dictionary to `pd.DataFrame()`:

- Each **dictionary key** becomes a column name.
- The values in each list become the **rows** of that column.

Let's create a DataFrame from our `data` dictionary.

In [32]:
# Create a DataFrame from the dictionary
purchases = pd.DataFrame(data)

purchases

,apples,oranges
0,3,0
1,2,3
2,0,7
3,1,2


The `Index` of this DataFrame was automatically created as `0, 1, 2, 3`.

We can replace this default index with an existing column using `.set_index()`.

> **Important:** By default, `.set_index()` returns a new DataFrame. It does not modify the original DataFrame unless we explicitly assign the result back or use `inplace=True`.

In [33]:
# Use the 'oranges' column as the index
orange_index_purchases = purchases.set_index('oranges')

# Display the new DataFrame
orange_index_purchases

,apples
oranges,
0,3
3,2
7,0
2,1


We can also create a DataFrame with our own index when we initialize it.

For this example, let's use customer names as the index.

In [34]:
# Create the DataFrame and use customer names as the index
purchases = pd.DataFrame(
    data,
    index=['Sarah', 'Tim', 'Lily', 'David']
)

# Display the DataFrame
purchases

,apples,oranges
Sarah,3,0
Tim,2,3
Lily,0,7
David,1,2


##### Select values using `.loc[]` and `.iloc[]`

We can select specific rows and columns from a DataFrame using:

- `.loc[]` to select data using **index and column labels**
- `.iloc[]` to select data using **integer positions**

For example, we can use the customer's name with `.loc[]`, or the customer's row position with `.iloc[]`.

In [35]:
# Select David's row using the index label 'David'
print(purchases.loc['David'])

print("--------------------")

# Select David's row using his integer position (row 3)
print(purchases.iloc[3])

apples     1
oranges    2
Name: David, dtype: int64
--------------------
apples     1
oranges    2
Name: David, dtype: int64


##### Select a single value using `.loc[]` and `.iloc[]`

We can also use `.loc[]` and `.iloc[]` to select a single value from a DataFrame.

- `.loc[row_label, column_label]` selects a value using labels.
- `.iloc[row_position, column_position]` selects a value using integer positions.

For example, we can find the number of oranges purchased by David using both approaches.

In [36]:
# Select the number of oranges David purchased using the row and column labels
oranges_david = purchases.loc['David', 'oranges']
print(oranges_david)

# Select the same value using David's row position (3) and the oranges column position (1)
oranges_david = purchases.iloc[3, 1]
print(oranges_david)

2
2


### Resetting the Index

So far, the customer names are being used as the DataFrame index.

What if we want to return to the default numbered index (`0, 1, 2, ...`) while keeping the customer names as a regular column?

We can use `.reset_index()` to move the current index into a new column.

In [37]:
# Reset the index and move the customer names into a new column
purchases = purchases.reset_index()

purchases

,index,apples,oranges
0,Sarah,3,0
1,Tim,2,3
2,Lily,0,7
3,David,1,2


### Rename the Generated Column

After resetting the index, the customer names are stored in a column called `index`.

Let's rename this column to `name` to make the DataFrame clearer and more descriptive.

### Modifying a DataFrame: `inplace=True` vs. Assignment

Many pandas methods return a modified copy of a DataFrame rather than changing the original DataFrame automatically.

There are two common ways to keep the result:

1. **Assign the result back to the DataFrame**
- purchases = purchases.rename(...)

2. Use inplace=True to modify the existing DataFrame directly
- purchases.rename(..., inplace=True)

Both approaches can produce the same result, but they work differently.

Important: When inplace=True is used, the method typically returns None, because the DataFrame itself has been modified.

In [38]:
# Return a modified DataFrame and assign it back to purchases
purchases = purchases.rename(columns={'index': 'name'})

purchases


,name,apples,oranges
0,Sarah,3,0
1,Tim,2,3
2,Lily,0,7
3,David,1,2


In [39]:
# Create a copy so that we can demonstrate inplace=True separately
purchases_copy = purchases.copy()

# Rename the column directly in the existing DataFrame
purchases_copy.rename(columns={'name': 'customer'}, inplace=True)

purchases_copy

,customer,apples,oranges
0,Sarah,3,0
1,Tim,2,3
2,Lily,0,7
3,David,1,2


### 1.3. DataFrame Manipulation

We can modify an existing DataFrame by adding, removing, or changing columns and rows.

#### Adding a Column

Suppose our store starts selling bananas.

We can add a new column by assigning a list of values to a new column name. The number of values must match the number of rows in the DataFrame.

In [40]:
# Add a new 'bananas' column with one value for each customer
purchases['bananas'] = [0, 1, 3, 3]

purchases

,name,apples,oranges,bananas
0,Sarah,3,0,0
1,Tim,2,3,1
2,Lily,0,7,3
3,David,1,2,3


#### Adding a Row

Suppose we have a new customer, Dan.

To add a row to an existing DataFrame, we can create a small DataFrame containing the new customer's information and combine it with the original DataFrame using `pd.concat()`.

> **Note:** `DataFrame.append()` was removed in pandas 2.0. We use `pd.concat()` instead.

In [41]:
# Create a DataFrame containing the new customer's information
new_row = pd.DataFrame([{
    'name': 'Dan',
    'apples': 2,
    'oranges': 2,
    'bananas': 0
}])

# Add the new row to the existing DataFrame
purchases = pd.concat([purchases, new_row], ignore_index=True)

purchases

,name,apples,oranges,bananas
0,Sarah,3,0,0
1,Tim,2,3,1
2,Lily,0,7,3
3,David,1,2,3
4,Dan,2,2,0


#### Finding the Maximum Value Across Columns

**Question:** What is the maximum number of items purchased by each customer, considering all fruit categories?

We want one maximum value for each row, so we need to aggregate **across columns**.

In pandas, `axis=1` performs the operation across the columns of each row.

In [42]:
# Find the maximum purchase across the fruit columns for each customer
purchases[['apples', 'oranges', 'bananas']].max(axis=1)

0    3
1    3
2    7
3    3
4    2
dtype: int64

**Question:** What is the highest number of items purchased for each fruit category across all customers?

We want one maximum value for each column, so we need to aggregate **down the rows**.

In pandas, `axis=0` performs the operation down each column.

In [43]:
# Find the maximum purchase for each fruit category across all customers
purchases[['apples', 'oranges', 'bananas']].max(axis=0)

apples     3
oranges    7
bananas    3
dtype: int64

### Understanding `axis`

Before moving on, let's check the shape of our DataFrame:


In [44]:
purchases.shape

(5, 4)

### Transpose a DataFrame

Transposing a DataFrame switches its rows and columns.

We can use the `.T` attribute or the `.transpose()` method. Here, we use `.transpose()` because it makes the operation explicit.


In [46]:
# Transpose the DataFrame: rows become columns and columns become rows
purchases_t = purchases.transpose()

# Display the transposed DataFrame
purchases_t

,0,1,2,3,4
name,Sarah,Tim,Lily,David,Dan
apples,3,2,0,1,2
oranges,0,3,7,2,2
bananas,0,1,3,3,0


In [48]:
# Let's check the shape again
purchases_t.shape

(4, 5)

## 2. Exploring an Imported DataFrame

So far, we created a small DataFrame ourselves.

In practice, however, data is often provided as a file. Pandas provides several functions for importing data from common file formats.

### 2.1 Reading Data from a CSV File

A **CSV (comma-separated values)** file stores tabular data as rows and columns.

Pandas provides `pd.read_csv()` to read a CSV file into a DataFrame.

In this tutorial, we will use a dataset containing baby-name records from the United States.

> **Note:** Make sure the CSV file is located in the same folder as this notebook, or provide the appropriate file path.

In [49]:
# Read the CSV file into a Pandas DataFrame
df_names = pd.read_csv('US_baby_names_2013-14.csv')

# Display the first five rows
df_names.head()

,Id,Name,Year,Gender,State,Count
0,13298,Emma,2013,F,AK,57
1,13299,Sophia,2013,F,AK,50
2,13300,Abigail,2013,F,AK,39
3,13301,Isabella,2013,F,AK,38
4,13302,Olivia,2013,F,AK,35


### Inspecting the DataFrame

After importing a dataset, one of the first things we should do is understand its structure.

The `.info()` method provides useful information about:

- the number of rows and columns
- column names
- the number of non-missing values
- the data type of each column
- approximate memory usage

Let's inspect our DataFrame.

In [50]:
# Display information about the DataFrame structure, including column names and data types
df_names.info()

<class 'pandas.DataFrame'>
RangeIndex: 186891 entries, 0 to 186890
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   Id      186891 non-null  int64
 1   Name    186891 non-null  str  
 2   Year    186891 non-null  int64
 3   Gender  186891 non-null  str  
 4   State   186891 non-null  str  
 5   Count   186891 non-null  int64
dtypes: int64(3), str(3)
memory usage: 8.6 MB


### Summary Statistics

The `.describe()` method provides summary statistics for numerical columns by default.

These statistics include:

- `count` — number of non-missing values
- `mean` — average value
- `std` — standard deviation
- `min` — minimum value
- `25%`, `50%`, `75%` — quartiles
- `max` — maximum value

Let's examine the numerical columns in our dataset.

In [51]:
# Display summary statistics for the numerical columns
df_names.describe()

,Id,Year,Count
count,1.868910e+05,186891.000000,186891.000000
mean,2.852127e+06,2013.503759,33.067692
std,1.652032e+06,0.499987,87.988787
min,1.329800e+04,2013.000000,5.000000
25%,1.325804e+06,2013.000000,7.000000
50%,2.816340e+06,2014.000000,11.000000
75%,4.347148e+06,2014.000000,26.000000
max,5.647426e+06,2014.000000,3451.000000


### Exploring Categorical Values

We can use `.unique()` to find the distinct values in a column.

For example, let's find all of the states represented in our dataset.

In [52]:
# Find the unique state abbreviations in the dataset
df_names['State'].unique()

<StringArray>
['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA',
 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS',
 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA',
 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']
Length: 51, dtype: str

### 2.2 Filtering a DataFrame Based on Conditions

Boolean filtering allows us to select only the rows that satisfy one or more conditions.

For example, suppose we want to find the **10 most popular male baby names in California in 2013**.

We can break this task into two steps:

1. Filter the DataFrame to keep only the relevant rows.
2. Sort the filtered data by the number of births and select the top 10.

In [53]:
# Keep only rows where the year is 2013
# and the state is California
# and the gender is male
df_M_Cali_2013 = df_names[
    (df_names['Year'] == 2013) &
    (df_names['State'] == 'CA') &
    (df_names['Gender'] == 'M')
]

# Display the filtered DataFrame
df_M_Cali_2013

,Id,Name,Year,Gender,State,Count
19288,704421,Jacob,2013,M,CA,2879
19289,704422,Ethan,2013,M,CA,2659
19290,704423,Daniel,2013,M,CA,2590
19291,704424,Jayden,2013,M,CA,2580
19292,704425,Matthew,2013,M,CA,2553
...,...,...,...,...,...,...
22174,707307,Zadkiel,2013,M,CA,5
22175,707308,Zavion,2013,M,CA,5
22176,707309,Zayed,2013,M,CA,5
22177,707310,Zixuan,2013,M,CA,5


The three conditions are combined using `&`, which means **AND**.

Notice that each condition is surrounded by parentheses. This is required when combining multiple Pandas conditions with `&`.

In [54]:
# Sort the filtered DataFrame by Count from largest to smallest
sorted_df_M_Cali_2013 = df_M_Cali_2013.sort_values(
    by='Count',
    ascending=False
)

# Select the first 10 rows
top_10_M_Cali_2013 = sorted_df_M_Cali_2013.head(10)

# Display the 10 most popular names
top_10_M_Cali_2013

,Id,Name,Year,Gender,State,Count
19288,704421,Jacob,2013,M,CA,2879
19289,704422,Ethan,2013,M,CA,2659
19290,704423,Daniel,2013,M,CA,2590
19291,704424,Jayden,2013,M,CA,2580
19292,704425,Matthew,2013,M,CA,2553
19293,704426,Noah,2013,M,CA,2544
19294,704427,Alexander,2013,M,CA,2376
19295,704428,Anthony,2013,M,CA,2227
19296,704429,Nathan,2013,M,CA,2077
19297,704430,David,2013,M,CA,2038


### 2.3 Using `groupby()` to Aggregate Data

The `groupby()` method allows us to split data into groups and then perform a calculation on each group.

For example, we can group the baby-name data by:

- `Year`
- `Gender`

We can then add the values in the `Count` column to find the total number of births for each year and gender.

**Question:** How many male and female babies were born in 2014?

In [55]:
# Group the data by year and gender
# Then select the Count column and calculate the total for each group
births_by_year_gender = (
    df_names
    .groupby(['Year', 'Gender'])['Count']
    .sum()
)

# Display the grouped totals
births_by_year_gender

Year  Gender
2013  F         1419351
      M         1647092
2014  F         1446259
      M         1667352
Name: Count, dtype: int64

The result has a **MultiIndex** because we grouped by two columns: `Year` and `Gender`.

For example, we can use `.loc[]` to select the results for a specific year.

In [56]:
# Select the results for the year 2014
births_by_year_gender.loc[2014]

Gender
F    1446259
M    1667352
Name: Count, dtype: int64

##### Combining Filtering and `groupby()`

We can combine several Pandas operations to answer more interesting questions.

**Question:** What are the most popular baby names containing the letter **"z"**?

First, we will:

1. Filter names containing the letter `"z"`.
2. Group the results by name.
3. Add the birth counts for each name.
4. Sort the names from most to least popular.

In [71]:
# Keep names containing the letter "z"
# case=False means that both "z" and "Z" are matched
# na=False means missing values are treated as not matching
names_with_z = df_names[
    df_names['Name'].str.contains('z', case=False, na=False)
]

# Group by name and calculate the total number of births
name_counts_z = (
    names_with_z
    .groupby('Name')['Count']
    .sum()
    .sort_values(ascending=False)
)

# Display the 10 most popular names containing "z"
name_counts_z.head(10)

Name
Elizabeth    18905
Zoey         14568
Zoe          11791
Zachary      10863
Mackenzie     8144
Ezra          6267
Hazel         4922
Ezekiel       4498
Mckenzie      4488
Zayden        4281
Name: Count, dtype: int64

The result is a **Series** because we selected only the `Count` column after grouping.

If we want the result to remain a DataFrame, we can use `as_index=False` and keep `Name` as a regular column.

### Resetting the index

After using `groupby('Name')`, the name becomes the DataFrame's index.

If we want `Name` back as a regular column, we can use `.reset_index()`.

Here we will use `inplace=True` to modify the existing DataFrame directly.

In [72]:
# Keep names containing the letter "z"
# case=False means that both "z" and "Z" are matched
# na=False means missing values are treated as not matching
names_with_z = df_names[
    df_names['Name'].str.contains('z', case=False, na=False)
]

# Group by name and calculate the total number of births
name_counts_z = (
    names_with_z
    .groupby('Name', as_index=False)['Count']
    .sum()
    .sort_values('Count', ascending=False)
)

# Display the 10 most popular names containing "z"
name_counts_z.head(10)

,Name,Count
146,Elizabeth,18905
704,Zoey,14568
699,Zoe,11791
511,Zachary,10863
350,Mackenzie,8144
163,Ezra,6267
190,Hazel,4922
157,Ezekiel,4498
379,Mckenzie,4488
615,Zayden,4281


## 2.4 Iteration and vectorization

Sometimes we want to create a new column based on conditions in existing columns.

For example, suppose we want to identify female baby names with at least 500 births.

A value of `"Yes"` should be assigned when both conditions are satisfied:

- `Gender` is `"F"`
- `Count` is greater than or equal to `500`

Otherwise, the value should be `"No"`.

We will first see how this can be done with iteration, and then compare it with a more natural Pandas approach.

In [73]:
# Work with the first 30,000 rows for this exercise
data = df_names.iloc[:30000].copy()

# Check the number of rows and columns
data.shape

(30000, 6)

In [74]:
%%time

# Create an empty list to store the results
high_female_birth = []

# Iterate through the rows of the DataFrame
for index, row in data.iterrows():

    # Check whether the name is female and has at least 500 births
    if (row['Count'] >= 500) and (row['Gender'] == 'F'):
        high_female_birth.append('Yes')
    else:
        high_female_birth.append('No')

# Add the results as a new column
data['HighFemaleBirth'] = high_female_birth

CPU times: user 355 ms, sys: 8.51 ms, total: 363 ms
Wall time: 390 ms


In [75]:
%%time

# Define a function that determines whether a row meets both conditions
def add_high_female_birth(row):
    if (row['Count'] >= 500) and (row['Gender'] == 'F'):
        return 'Yes'
    return 'No'

# Apply the function to each row
data['HighFemaleBirth'] = data.apply(
    add_high_female_birth,
    axis=1
)

CPU times: user 79.5 ms, sys: 4.31 ms, total: 83.8 ms
Wall time: 84.8 ms


In [76]:
%%time
# OPTION 2: Simple for loop using .iterrows()

# initalize empty list
HighFemaleBirth = []

for index, row in data.iterrows():
    Count = row['Count']
    Gender = row['Gender']
    
    if (Count >=500) and (Gender == 'F'):
        HighFemaleBirth.append('Yes')
    else:
        HighFemaleBirth.append('No')
        
data['HighFemaleBirth'] = HighFemaleBirth

CPU times: user 389 ms, sys: 4.34 ms, total: 393 ms
Wall time: 393 ms


In [77]:
%%time

# Create the new column using Boolean conditions
data['HighFemaleBirth'] = np.where(
    (data['Count'] >= 500) & (data['Gender'] == 'F'),
    'Yes',
    'No'
)

CPU times: user 5.41 ms, sys: 3.04 ms, total: 8.46 ms
Wall time: 7.48 ms


### Why prefer this approach?

The previous examples process the DataFrame **row by row**.

Pandas is designed to work with entire columns at once. This is called **vectorization**.

The vectorized approach is usually:

- shorter,
- easier to read,
- and faster for large datasets.

When working with Pandas, we generally prefer vectorized operations when possible.

In [79]:
# Count how many rows have each value
data['HighFemaleBirth'].value_counts()

HighFemaleBirth
No     29852
Yes      148
Name: count, dtype: int64

In [80]:
# Display rows where the condition was satisfied
data[data['HighFemaleBirth'] == 'Yes']

,Id,Name,Year,Gender,State,Count,HighFemaleBirth
6837,307228,Sophia,2013,F,AZ,604,Yes
11353,557545,Sophia,2013,F,CA,3451,Yes
11354,557546,Isabella,2013,F,CA,2783,Yes
11355,557547,Mia,2013,F,CA,2592,Yes
11356,557548,Emma,2013,F,CA,2478,Yes
...,...,...,...,...,...,...,...
15374,561566,Valerie,2014,F,CA,534,Yes
15375,561567,Ruby,2014,F,CA,531,Yes
15376,561568,Claire,2014,F,CA,520,Yes
15377,561569,Ariel,2014,F,CA,507,Yes


## 3. Practice Challenge

Use the `df_names` DataFrame to answer the following questions.

### Challenge 1 — Filtering

Find the 10 most popular **female baby names in California (CA) in 2014**.

---

### Challenge 2 — Grouping and aggregation

How many male and female babies were born in **2013**?

---

### Challenge 3 — Grouping and sorting

Find the **10 names with the highest total number of births** across all states and years in the dataset.

---

### Challenge 4 — String filtering and grouping

Find the **10 most popular names containing the letter `"x"`**.

Remember that names may contain either uppercase or lowercase `"X"`.

---

### Challenge 5 — Creating a new column

Create a new column called `HighBirthCount`.

For each row:

- Assign `"Yes"` if `Count` is greater than or equal to `500`.
- Assign `"No"` otherwise.

Try to solve this using `.apply()` and a function.

---

### Challenge 6 — Think about the result

After creating `HighBirthCount`, determine how many rows have:

- `HighBirthCount == "Yes"`
- `HighBirthCount == "No"`

### Reminder

Try to solve the challenges using the Pandas techniques introduced in this tutorial:

- Boolean filtering
- `.loc[]` and `.iloc[]`
- `.sort_values()`
- `.groupby()`
- `.sum()`
- `.head()`
- `.str.contains()`
- `.apply()`

The goal is to practice combining these techniques rather than learning new Pandas functionality.